# 第10周 · 作业（预计约7小时）

| 部分 | 内容 | 建议时间 |
| --- | --- | --- |
| A | 研究实验 2：TextCNN 的卷积核宽度（4 种宽度 × 3 个种子 + TF-IDF 基线） | 4 小时 |
| B | Capstone 一页提案 | 2 小时 |
| C | 代码讲解段落 | 1 小时 |

**规则**：可以使用 AI 编程助手，但**必须附上你自己写的"代码讲解"段落**（C 部分）——讲清代码做了什么、shape 怎么变、你如何确认它是对的。批改时会随机抽一段请你口头讲解。

**算力**：A 部分共 12 次训练。Colab 请切换到 T4 GPU（全部约 1–2 分钟）；纯 CPU 也能跑完（老师的电脑上约 4–6 分钟）。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from zero2researcher import data, text, experiments
from zero2researcher.checks import week10 as ck

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

## A · 研究实验 2：卷积核宽度重要吗？

**研究问题**：在 IMDB 课程子集上，单一宽度 TextCNN 的卷积核宽度 $k$ 如何影响测试准确率？

第 6 周你做过一个"兄弟实验"：把 TF-IDF 的 n-gram 范围从 (1, 1) 扩到 (1, 3)，测试准确率**没有**提升（3,000 条训练子集上约 0.855 → 0.847 → 0.842）。课上我们看到，宽度为 $k$ 的卷积核 ≈ 一个可学习的 $k$-gram 检测器。那么在 CNN 里，"看更长的短语"会有帮助吗？

**设计**（一次只改一处）：

- **自变量**：卷积核宽度 $k \in \{2, 3, 5, 7\}$——每个模型只用**一种**宽度的 64 个核；
- **控制变量**（全部冻结）：`text.imdb_arrays(max_len=256, max_vocab=20000)`；从零训练的 `Embedding(V, 64, padding_idx=0)`；ReLU + 全局 max pooling；dropout 0.5；Adam lr 1e-3；batch 64；**5 个 epoch**；测试集固定为全部 2,000 条；
- **种子**：每个 $k$ 跑 seed ∈ {0, 1, 2}，每次 run 的开头调用 `experiments.set_seed(seed)`（共 12 次训练）；
- **基线**：TF-IDF + 逻辑回归（第 6 周的配置；它是确定性的，跑一次即可）；
- **报告**：每个 $k$ 的测试准确率**均值 ± 标准差**（`experiments.format_result`），以及一张画了基线水平线的误差条图。

**纪律**：测试集只在每次训练**结束后**看一次，不许按测试集挑 epoch。

### 第 0 步：数据与基线

In [ ]:
Xtr, ytr, Xte, yte, vocab = text.imdb_arrays(max_len=256, max_vocab=20000)
train_df, test_df = data.load_imdb("train"), data.load_imdb("test")
print("训练:", Xtr.shape, "  测试:", Xte.shape, "  词表:", len(vocab))

# 在这里写你的代码
raise NotImplementedError
print(f"TF-IDF + LR 基线: {tfidf_acc:.4f}")
ck.tfidf_baseline(tfidf_acc)

### 第 1 步：单一宽度的 TextCNN

照着课上的 `TextCNN` 改：只有**一个** `Conv1d`，宽度由参数 `k` 决定；其余结构与课上相同（Embedding → **转置** → Conv1d → ReLU → 全局 max pooling → dropout → Linear）。写完用 `ck.textcnn_shapes` 验收最窄与最宽的两种宽度。

In [ ]:
class SingleWidthCNN(nn.Module):
    def __init__(self, vocab_size, k=3, emb_dim=64, n_filters=64, n_classes=2, dropout=0.5):
        super().__init__()
        # 在这里写你的代码
        raise NotImplementedError

    def forward(self, x):                                   # x: (N, L) 的词 id
        # 在这里写你的代码
        raise NotImplementedError


for k in (2, 7):
    ck.textcnn_shapes(SingleWidthCNN(len(vocab), k=k))

### 第 2 步：一次 run = 一个 $(k, \text{seed})$

写 `run(k, seed)`：设种子 → 建训练 DataLoader 与模型 → 训练 `EPOCHS` 个 epoch → 返回测试准确率。训练与评估直接用第 8 周写过的两个函数（下面已给出）。

为什么 `set_seed` 要放在建 DataLoader 和模型**之前**？因为种子要同时管住三样随机性：参数初始化、每个 epoch 的打乱顺序、dropout 的随机掩码。

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = criterion(model(xb), yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
        n += len(xb)
    return total / n


def evaluate(model, loader, device):
    model.eval()
    correct, n = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(dim=1) == yb).sum().item()
            n += len(yb)
    return correct / n


EPOCHS = 5
train_ds = TensorDataset(torch.tensor(Xtr), torch.tensor(ytr))
test_loader = DataLoader(TensorDataset(torch.tensor(Xte), torch.tensor(yte)), batch_size=256)


def run(k, seed):
    """训练一个宽度为 k 的 SingleWidthCNN，返回测试准确率"""
    # 在这里写你的代码
    raise NotImplementedError


### 第 3 步：跑完 12 次，边跑边看耗时

把结果存成 `results[k] = [seed 0 的准确率, seed 1 的, seed 2 的]`。每次 run 打印一行用时——**第一行出来你就能估算总预算**（核越宽越慢，k = 7 大约是 k = 2 的 2–2.5 倍）。如果估出来太久，先别硬跑：换 GPU，或者回来调整预算并在"局限"里写明。

In [ ]:
results = {}
t_start = time.time()
# 在这里写你的代码
raise NotImplementedError
print(f"总用时 {(time.time() - t_start) / 60:.1f} 分钟")
ck.kernel_width_results(results)

### 第 4 步：表格与误差条图

表格每行一个设置：TextCNN 的 4 种宽度用 `experiments.format_result(accs)` 写成"均值 ± 标准差 (n=3)"，最后一行是 TF-IDF 基线。图的 x 轴是核宽，y 轴是测试准确率（误差条 = 标准差），用一条水平虚线画出基线。**标题写结论**（例如"更宽的核……"），不要写"Accuracy vs k"。

In [ ]:
# 在这里写你的代码
raise NotImplementedError


### 第 5 步：实验记录

把实验写成**实验记录 #2**（格式与第 6 周相同）。"解释"一栏请回答两个问题：

1. 你的结果与第 6 周的 n-gram 实验（TF-IDF 上 (1,1) ≈ 0.855、(1,2) ≈ 0.847、(1,3) ≈ 0.842）是否一致？"看更长的短语"在两种模型里各自起了什么作用？
2. 课上看到 TextCNN 的核在找 *waste of time*、*great performance* 这类短语。判断情感需要的短语一般有多长？这能解释你的结果吗？

"局限"一栏至少写一条关于**训练预算**的：所有宽度都只训练了 5 个 epoch——如果宽核收敛得更慢，这个固定的预算偏袒了谁？你会怎么设计实验排除这个解释？

结论只能是三种之一：显著更好 / 显著更差 / 差异小于噪声——并给出依据（均值差与标准差的比较）。

## 实验记录 #2 · 日期：____

- 研究问题：
- 假设（预期结果，以及什么结果会推翻它）：
- 自变量（这次只改变什么）：
- 控制变量（哪些保持不变；哪些干扰变量单独调优）：
- 基线：
- 数据与划分方式（如何避免泄漏）：
- 随机种子：
- 结果（数值：均值±标准差；图：链接或插图）：
- 解释：
- 局限：
- 下一步：

## B · Capstone 一页提案

从第 8 周作业 C 的 3 个候选里挑**一个**，写成一页纸的提案（模板取自学生手册 7.2 节）。"一页纸"的意思是：每一栏 1–3 句话，让导师两分钟内看懂你要回答什么问题、凭什么能做成。第 11 周的一对一会逐条过这份提案。

动笔前对照三条检查：

- **研究问题**一句话说得完，而且答案是一个**数字比较**（"X 是否比 Y 高 / 低"），而不是"探索 X"；
- **基线**是今天就能跑起来的东西（多数类、逻辑回归、随机森林、TF-IDF……）——本周的教训：强基线可能比你的模型还强，这正是要提前知道的；
- **备选公开数据题**真的用公开数据：万一实验室的数据拿不到，你第二天就能开工。

示例（只示范粒度，不要照抄）：

> **题目**：小样本下，k-mer 计数 + 逻辑回归是否不输一维 CNN？
>
> - 研究问题：在几千条带标签的蛋白质序列上，一维 CNN 的分类准确率是否显著高于 3-mer 计数 + 逻辑回归？
> - 被翻转的默认假设：序列数据上，深度模型理所当然优于手工特征。
> - 可检验的预测：3 个种子下两者准确率之差 < 2 个百分点；训练数据增加到 4 倍后，CNN 领先超过 3 个百分点。
> - 数据：某个公开的蛋白质亚细胞定位数据集，只取其中两类；已下载。
> - 基线模型：多数类；3-mer 计数 + 逻辑回归。
> - 预期结果与意义：若预测成立，说明小样本的实验室数据先用 k-mer 基线就够，CNN 的优势需要数据量来兑现——与本周 IMDB 上的发现同构。
> - 备选公开数据题：同一问题换成 IMDB 子集上的 TextCNN vs TF-IDF，比较训练集 1k / 5k 两种规模。

# 题目：

- 研究问题（一句话）：
- 被翻转的默认假设（bit flip）：
- 可检验的预测：
- 数据（来源、规模、是否已拿到）：
- 基线模型：
- 预期结果与意义：
- 备选公开数据题：

## C · 代码讲解（AI 助手使用说明）

本周起可以使用 AI 编程助手。作为交换，请为 A 部分的 `SingleWidthCNN.forward` 和 `run` **各写一段你自己的讲解**（每段 4–6 句话）：

- 这段代码按什么顺序做了哪几件事；
- 关键张量的 shape 每一步怎么变——从一个 batch 的词 id `(64, 256)` 开始写起；
- 你是怎么确认它是对的（跑过哪个自查、打印过什么、发现并修过什么 bug）；
- 如果用了 AI：你问了它什么，它给的代码里哪一部分你改掉了，为什么。

**`SingleWidthCNN.forward` 的讲解**：

（在这里写）

**`run` 的讲解**：

（在这里写）

---

**下周预告**：第 11 周 · Attention 与 Transformer。TextCNN 的每个核只能看 $k$ 个相邻的词；注意力让每个词**直接**看到整条影评里的每一个词。你将从零写出缩放点积注意力和多头注意力，搭一个 Transformer 编码器，在同一份 IMDB 上和 TF-IDF、TextCNN 比一比——别忘了带上本周课上（lecture S3 末尾）保存的 `outputs/imdb_results.csv`——第 11 周作业会读它、再追加一行。